<a href="https://colab.research.google.com/github/rafaellopesdesa/hnsbi-toolkit/blob/main/examples/notebooks/neural_importance_sampling_asimov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural importance-sampling Asimov construction

This is the native counterpart of Exercise 6. It reuses the verified nominal reference and ratio artifacts from the hybrid notebook when available, trains a scan-aware proposal, and restores the six earlier diagnostics: proposal/target closure, defensive reweighting back to the reference, a high-statistics benchmark scan, repeated convergence, repeated small scans, and the equal-cost $q_{0,A}$ comparison.


In [ ]:
import importlib
from importlib.metadata import PackageNotFoundError, version as installed_version
from pathlib import Path
import os, signal, subprocess, sys

def distribution_version(name):
    try:
        return installed_version(name)
    except PackageNotFoundError:
        return None

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    loaded_versions = {
        name: getattr(sys.modules.get(name), '__version__', None)
        for name in ('numpy', 'jax', 'jaxlib')
    }
    jax_was_loaded = any(
        name == 'jax' or name.startswith(('jax.', 'jaxlib', 'jax_plugins'))
        for name in sys.modules
    )
    ROOT = Path('/content/drive/MyDrive/hsbi-toolkit')
    REPO = ROOT / 'hnsbi-toolkit'
    ROOT.mkdir(parents=True, exist_ok=True)
    if not (REPO / '.git').is_dir():
        subprocess.run(['git', 'clone', 'https://github.com/rafaellopesdesa/hnsbi-toolkit.git', str(REPO)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[lhc,flows]'], check=True)

    # Colab installs JAX's CUDA plugin separately from jaxlib. If an
    # earlier dependency resolution changed jaxlib, realign the plugin
    # before JAX discovers it; mixed PJRT versions fail at execution.
    jaxlib_version = distribution_version('jaxlib')
    plugin_extras = {
        'jax-cuda12-plugin': 'cuda12-local',
        'jax-cuda13-plugin': 'cuda13-local',
    }
    repaired_plugins = []
    for plugin, extra in plugin_extras.items():
        plugin_version = distribution_version(plugin)
        if plugin_version is not None and plugin_version != jaxlib_version:
            print(f'Aligning {plugin} {plugin_version} with jaxlib {jaxlib_version}.')
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                 f'jax[{extra}]=={jaxlib_version}'],
                check=True,
            )
            if distribution_version(plugin) != jaxlib_version:
                raise RuntimeError(f'Could not align {plugin} with jaxlib.')
            repaired_plugins.append(plugin)

    changed_loaded_packages = [
        name for name, loaded in loaded_versions.items()
        if loaded is not None and loaded != distribution_version(name)
    ]
    if changed_loaded_packages or (repaired_plugins and jax_was_loaded):
        reasons = changed_loaded_packages + repaired_plugins
        print(
            f'Updated {", ".join(dict.fromkeys(reasons))}. '
            'Restarting the Colab runtime once to load a consistent NumPy/JAX stack. '
            'After it reconnects, run this setup cell again.',
            flush=True,
        )
        os.kill(os.getpid(), signal.SIGKILL)
else:
    REPO = Path.cwd()
    if not (REPO / 'pyproject.toml').exists():
        REPO = Path.cwd().parents[1]
os.chdir(REPO)
source_root = (REPO / 'src').resolve()
expected_hnsbi = source_root / 'hnsbi' / '__init__.py'
notebook_helpers = str((REPO / 'examples' / 'notebooks').resolve())
if notebook_helpers not in sys.path:
    # Append so the LHC example's generate_distributions.py remains canonical.
    sys.path.append(notebook_helpers)
for import_dir in (source_root, REPO / 'examples' / 'lhc_analysis'):
    import_path = str(import_dir.resolve())
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

# An editable install writes a .pth file, but the running interpreter does
# not process a newly created .pth until its next start. Remove any namespace
# placeholder cached before the source tree became importable.
loaded_hnsbi = sys.modules.get('hnsbi')
loaded_file = getattr(loaded_hnsbi, '__file__', None)
if loaded_hnsbi is not None and (
    loaded_file is None or Path(loaded_file).resolve() != expected_hnsbi
):
    for module_name in tuple(sys.modules):
        if module_name == 'hnsbi' or module_name.startswith('hnsbi.'):
            sys.modules.pop(module_name, None)
importlib.invalidate_caches()
hnsbi_package = importlib.import_module('hnsbi')
actual_hnsbi = Path(hnsbi_package.__file__).resolve()
if actual_hnsbi != expected_hnsbi or not hasattr(hnsbi_package, 'Project'):
    raise ImportError(
        f'Expected hnsbi with Project from {expected_hnsbi}, got {actual_hnsbi}'
    )


## Reuse the nominal hybrid artifacts

Exercise 6 should build on Exercise 5, not overwrite it. This cell keeps any sample at or above the hybrid notebook's event counts and loads the checksummed flow, ratio ensembles, and ratio normalizer when present. If the notebook is run standalone, it generates the same high-statistics samples and trains the missing nominal artifacts once.


In [ ]:
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch

from generate_distributions import EXPECTED_YIELDS, generate
from hnsbi import Project
from hnsbi.flows import ReferenceFlow
from hnsbi.intensity import RatioNormalizer
from hnsbi.native_ratios import load_native_ratio_ensemble
from utils_lhc_validation import (
    verify_reuse_provenance,
    write_reuse_provenance,
)

EXAMPLE = REPO / 'examples' / 'lhc_analysis'
DATA = EXAMPLE / 'data'
TARGET_ROWS = {
    'signal.parquet': 120_000,
    'background.parquet': 300_000,
    'reference.parquet': 400_000,
}

def parquet_rows(path):
    return pq.ParquetFile(path).metadata.num_rows if path.is_file() else 0

observed_rows = {
    name: parquet_rows(DATA / name)
    for name in TARGET_ROWS
}
regenerate = any(
    observed_rows[name] < required
    for name, required in TARGET_ROWS.items()
)
if regenerate:
    print('Generating the shared high-statistics nominal samples.')
    generate(
        DATA,
        signal_events=TARGET_ROWS['signal.parquet'],
        background_events=TARGET_ROWS['background.parquet'],
        reference_events=TARGET_ROWS['reference.parquet'],
    )
else:
    print('Reusing existing samples:', observed_rows)

project = Project.load(EXAMPLE / 'analysis.yaml')
FEATURES = tuple(project.config.features)
ARTIFACTS = EXAMPLE / 'artifacts'
reference_checkpoint = ARTIFACTS / 'reference' / 'reference_flow.pt'
ratio_manifests = {
    sample: ARTIFACTS / 'ratios' / sample / 'ratio_ensemble.manifest.json'
    for sample in ('signal', 'background')
}
normalizer_path = ARTIFACTS / 'ratios' / 'ratio_normalization.json'
nominal_data = {
    sample: DATA / f'{sample}.parquet'
    for sample in ('signal', 'background', 'reference')
}
nominal_artifacts = {
    'reference': reference_checkpoint.with_suffix(
        reference_checkpoint.suffix + '.manifest.json'
    ),
    'ratio-signal': ratio_manifests['signal'],
    'ratio-background': ratio_manifests['background'],
    'normalizer': normalizer_path.with_suffix(
        normalizer_path.suffix + '.manifest.json'
    ),
}
nominal_provenance = ARTIFACTS / 'nominal_training_provenance.json'
provenance_ok, reuse_reasons = verify_reuse_provenance(
    nominal_provenance,
    configuration_path=EXAMPLE / 'analysis.yaml',
    data_paths=nominal_data,
    artifact_paths=nominal_artifacts,
)
reusable = not regenerate and provenance_ok

if reusable:
    print('Loading verified Exercise-5 nominal artifacts.')
    if torch.cuda.is_available():
        flow_device = 'cuda'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        flow_device = 'mps'
    else:
        flow_device = 'cpu'
    reference = ReferenceFlow.load(
        reference_checkpoint,
        device=flow_device,
        expected_features=FEATURES,
    )
    ratio_evaluators = {
        sample: load_native_ratio_ensemble(
            manifest,
            expected_features=FEATURES,
        )
        for sample, manifest in ratio_manifests.items()
    }
    ratio_normalizer = RatioNormalizer.load(normalizer_path)
else:
    if regenerate:
        reuse_reasons = ('nominal samples were regenerated',)
    print(
        'Training nominal artifacts because the reusable bundle is unavailable: '
        + '; '.join(reuse_reasons)
    )
    reference_artifacts = project.train_reference()
    reference = reference_artifacts.training.flow
    ratio_artifacts = project.train_ratios(
        reference,
        normalization_events=40_000,
        seed=20260729,
    )
    ratio_evaluators = ratio_artifacts.evaluators
    ratio_normalizer = ratio_artifacts.normalizer
    nominal_artifacts = {
        'reference': reference_artifacts.checkpoint_manifest,
        'ratio-signal': ratio_artifacts.training['signal'].manifest_path,
        'ratio-background': ratio_artifacts.training['background'].manifest_path,
        'normalizer': ratio_artifacts.normalizer_manifest,
    }
    write_reuse_provenance(
        nominal_provenance,
        configuration_path=EXAMPLE / 'analysis.yaml',
        data_paths=nominal_data,
        artifact_paths=nominal_artifacts,
        metadata={
            'features': list(FEATURES),
            'rows': {
                name.removesuffix('.parquet'): required
                for name, required in TARGET_ROWS.items()
            },
        },
    )

print('reference device:', reference.device)
print('ratio normalizers:', ratio_normalizer.means)

## Train and validate the proposal

The proposal is trained on the scan-wide influence amplitude $A(x)$. Sampling uses the defensive density

$$
g_\epsilon(x)=(1-\epsilon)g_\psi(x)+\epsilon q_\phi(x),
$$

which guarantees $q_\phi/g_\epsilon\leq 1/\epsilon$. The likelihood-convergence study fixes the three new nuisances at their nominal generating values, matching the scientific question in the original Exercise 6.


In [ ]:
NOMINAL_POINT = {
    'mu': 1.0,
    'response': 0.0,
    'resolution': 0.0,
    'theory': 0.0,
}
nis = project.train_nis_asimov(
    reference=reference,
    ratios=ratio_evaluators,
    truth_point=NOMINAL_POINT,
    asimov_point=NOMINAL_POINT,
)
print('raw count:', nis.asimov.raw_count)
print('ESS:', nis.asimov.ess)
print('validation:', nis.validation_provenance)
print('workspace-ready arrays:', nis.asimov_array_paths)


## Native proposal-flow validation

The toolkit first checks that the proposal flow reproduces its weighted training target. These compact feature, log-density, correlation, and pairwise diagnostics are additional to the six Exercise-6 plots below.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from hnsbi.flow_diagnostics import (
    plot_correlation_closure,
    plot_feature_closure,
    plot_log_prob_closure,
    plot_pairwise_closure,
)
from utils_plotting import export_standalone_figure_script

DIRECT_COLOR = '#d55e00'
NIS_COLOR = '#0072b2'
FIGURE_DIR = EXAMPLE / 'artifacts' / 'notebook_figures' / 'nis'
FIGURE_SCRIPT_DIR = FIGURE_DIR / 'scripts'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def export_nis_figure(figure, name):
    figure.savefig(FIGURE_DIR / f'{name}.png', dpi=160, bbox_inches='tight')
    return export_standalone_figure_script(
        figure,
        script_name=name,
        output_dir=FIGURE_SCRIPT_DIR,
    )

for name, plotter in (
    ('nis_native_feature_closure', plot_feature_closure),
    ('nis_native_log_prob_closure', plot_log_prob_closure),
    ('nis_native_correlation_closure', plot_correlation_closure),
    ('nis_native_pairwise_closure', plot_pairwise_closure),
):
    fig = plotter(nis.validation)
    export_nis_figure(fig, name)
    plt.show()


## Proposal versus variance-optimal target

On a fresh reference-flow sample, the centered relation

$$
\log\frac{g_\psi(x)}{q_\phi(x)}
\simeq \log A(x)+\text{constant}
$$

tests whether the learned proposal follows the local scan influence rather than merely matching a few feature projections.


In [ ]:
from hnsbi.nis import scan_influence_amplitude
from hnsbi.nis_diagnostics import (
    compare_nis_feature_closure,
    compare_nis_target,
    plot_nis_feature_closure,
    plot_nis_target_closure,
)

nis_configuration = project.config.frequentist['nis']
validation_rng = np.random.default_rng(20260731)
validation_values = reference.sample(30_000, rng=validation_rng)
validation_ratios = {
    sample: evaluator(validation_values)
    for sample, evaluator in ratio_evaluators.items()
}
validation_amplitude = scan_influence_amplitude(
    project.intensity_model(),
    validation_ratios,
    truth_point=NOMINAL_POINT,
    design_points=nis_configuration['design_points'],
    event_values=validation_values,
)
validation_amplitude = np.clip(
    validation_amplitude,
    nis.design.diagnostics['amplitude_floor'],
    nis.design.diagnostics['amplitude_ceiling'],
)
log_proposal_over_reference = (
    nis.flow_training.flow.log_prob(validation_values)
    - reference.log_prob(validation_values)
)
target_closure = compare_nis_target(
    validation_amplitude,
    log_proposal_over_reference,
)
print(target_closure.to_dict())
fig = plot_nis_target_closure(
    target_closure,
    rng=np.random.default_rng(20260732),
)
export_nis_figure(fig, 'proposal_target_closure')
plt.show()


## Defensive proposal reweighted back to the reference

This is a direct closure of the importance identity. Fresh $g_\epsilon$ events are weighted by $q_\phi/g_\epsilon$ and compared with an independent $q_\phi$ sample. The weights here are reference integration weights, not the physical Asimov weights.


In [ ]:
reference_closure_values = reference.sample(30_000, rng=validation_rng)
proposal_closure_values, proposal_log_q_over_g = (
    nis.defensive_proposal.sample_with_reference_log_weight(
        30_000,
        rng=validation_rng,
    )
)
feature_closure = compare_nis_feature_closure(
    reference_closure_values,
    proposal_closure_values,
    proposal_log_q_over_g,
    features=FEATURES,
)
print(feature_closure.to_dict())
fig = plot_nis_feature_closure(feature_closure, columns=5)
export_nis_figure(fig, 'importance_reweighted_reference_closure')
plt.show()

## Repeated direct-versus-NIS Asimov study

A high-statistics direct-reference quadrature defines the learned-model benchmark. Every smaller direct or defensive sample refits its process-ratio normalizers on its own support, preserving the exact truth-point score closure. `quick` produces all figures with modest runtime; `paper` restores the much heavier Exercise-6 statistics.


In [ ]:
from hnsbi.nis import NISAsimovBuilder
from utils_lhc_validation import asimov_mu_scan

STUDY_PROFILE = 'quick'  # 'quick' or 'paper'
if STUDY_PROFILE == 'paper':
    BENCHMARK_EVENTS = 1_000_000
    SAMPLE_SIZES = (512, 1_024, 2_048, 4_096, 8_192, 16_384, 32_768)
    REPETITIONS = 64
else:
    BENCHMARK_EVENTS = 60_000
    SAMPLE_SIZES = (512, 1_024, 2_048, 4_096)
    REPETITIONS = 8
SHOWCASE_SIZE = 2_048
MU_SCAN = np.linspace(0.0, 3.0, 61)

direct_builder = project.asimov_builder(
    reference=reference,
    ratios=ratio_evaluators,
)
nis_builder = NISAsimovBuilder(
    proposal=nis.defensive_proposal,
    ratios=ratio_evaluators,
    intensity=project.intensity_model(),
    features=FEATURES,
)

benchmark = direct_builder.build(
    NOMINAL_POINT,
    n_events=BENCHMARK_EVENTS,
    seed=20260800,
    normalization='sample',
)
benchmark_scan = asimov_mu_scan(
    benchmark,
    MU_SCAN,
    signal_yield=EXPECTED_YIELDS['signal'],
    background_yield=EXPECTED_YIELDS['background'],
)
q0_index = int(np.argmin(np.abs(MU_SCAN)))
benchmark_q0 = float(benchmark_scan[q0_index])
print(
    f'benchmark q0,A={benchmark_q0:.6f}; '
    f'physical-weight ESS={benchmark.ess:,.0f}/{BENCHMARK_EVENTS:,}'
)

fig, axis = plt.subplots(figsize=(6.5, 4.8))
axis.plot(MU_SCAN, benchmark_scan, color='black', lw=2.4)
axis.axvline(1.0, color='0.5', ls='--', lw=1.2)
axis.set(
    xlabel=r'$\mu$',
    ylabel=r'$t_A(\mu)$',
    title='High-statistics hybrid-model Asimov benchmark',
)
fig.tight_layout()
export_nis_figure(fig, 'asimov_benchmark_scan')
plt.show()

def quadrature_ess(result):
    weights = np.asarray(result.reference_weights, dtype=np.float64)
    return float(np.sum(weights) ** 2 / np.sum(weights**2))

rows = []
showcase_scans = {'Direct reference': [], 'Neural importance': []}
seed_sequence = np.random.SeedSequence(20260801)
child_seeds = iter(
    seed_sequence.generate_state(
        2 * len(SAMPLE_SIZES) * REPETITIONS,
        dtype=np.uint32,
    )
)
for sample_size in SAMPLE_SIZES:
    for repetition in range(REPETITIONS):
        direct = direct_builder.build(
            NOMINAL_POINT,
            n_events=sample_size,
            seed=int(next(child_seeds)),
            normalization='sample',
        )
        efficient = nis_builder.build(
            NOMINAL_POINT,
            n_events=sample_size,
            seed=int(next(child_seeds)),
        )
        for method, result in (
            ('Direct reference', direct),
            ('Neural importance', efficient),
        ):
            curve = asimov_mu_scan(
                result,
                MU_SCAN,
                signal_yield=EXPECTED_YIELDS['signal'],
                background_yield=EXPECTED_YIELDS['background'],
            )
            rows.append(
                {
                    'method': method,
                    'sample_size': sample_size,
                    'repetition': repetition,
                    'q_zero': float(curve[q0_index]),
                    'scan_squared_error': float(
                        np.mean((curve - benchmark_scan) ** 2)
                    ),
                    'ess': quadrature_ess(result),
                }
            )
            if (
                sample_size == SHOWCASE_SIZE
                and len(showcase_scans[method]) < 8
            ):
                showcase_scans[method].append(curve)

study_results = pd.DataFrame(rows)
summary_rows = []
for (method, sample_size), group in study_results.groupby(
    ['method', 'sample_size'],
    sort=False,
):
    q0 = group['q_zero'].to_numpy()
    summary_rows.append(
        {
            'method': method,
            'sample_size': int(sample_size),
            'q0_bias': float(np.mean(q0 - benchmark_q0)),
            'q0_standard_deviation': float(np.std(q0, ddof=1)),
            'q0_rmse': float(
                np.sqrt(np.mean((q0 - benchmark_q0) ** 2))
            ),
            'scan_rmse': float(
                np.sqrt(np.mean(group['scan_squared_error']))
            ),
            'mean_ess': float(np.mean(group['ess'])),
        }
    )
study_summary = pd.DataFrame(summary_rows)
direct_variance = (
    study_summary[study_summary['method'] == 'Direct reference']
    .set_index('sample_size')['q0_standard_deviation']
    .pow(2)
)
nis_variance = (
    study_summary[study_summary['method'] == 'Neural importance']
    .set_index('sample_size')['q0_standard_deviation']
    .pow(2)
)
variance_gain = direct_variance / nis_variance
study_summary['variance_reduction'] = [
    (
        variance_gain.loc[row.sample_size]
        if row.method == 'Neural importance'
        else np.nan
    )
    for row in study_summary.itertuples()
]
display(study_summary)

## Convergence, repeated scans, and equal-cost comparison

The first figure separates discovery-statistic error from whole-scan error and reports the approximate event-saving factor. The next two expose the distribution hidden by an average: individual small-sample scans and the equal-cost $q_{0,A}$ spread.


In [ ]:
colors = {
    'Direct reference': DIRECT_COLOR,
    'Neural importance': NIS_COLOR,
}
markers = {'Direct reference': 'o', 'Neural importance': 's'}
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
for method, group in study_summary.groupby('method', sort=False):
    group = group.sort_values('sample_size')
    axes[0].plot(
        group['sample_size'],
        group['q0_rmse'],
        marker=markers[method],
        color=colors[method],
        lw=2,
        label=method,
    )
    axes[1].plot(
        group['sample_size'],
        group['scan_rmse'],
        marker=markers[method],
        color=colors[method],
        lw=2,
        label=method,
    )
nis_summary = study_summary[
    study_summary['method'] == 'Neural importance'
].sort_values('sample_size')
axes[2].plot(
    nis_summary['sample_size'],
    nis_summary['variance_reduction'],
    marker='D',
    color=NIS_COLOR,
    lw=2,
)
axes[2].axhline(1.0, color='black', ls='--', lw=1.2)
for axis in axes[:2]:
    axis.set_xscale('log', base=2)
    axis.set_yscale('log')
    axis.grid(alpha=0.25)
    axis.legend()
axes[2].set_xscale('log', base=2)
axes[2].grid(alpha=0.25)
axes[0].set(
    xlabel='Number of Asimov points',
    ylabel=r'RMSE of $q_{0,A}$',
    title='Discovery statistic',
)
axes[1].set(
    xlabel='Number of Asimov points',
    ylabel=r'RMS error over $t_A(\mu)$',
    title='Complete likelihood scan',
)
axes[2].set(
    xlabel='Number of Asimov points',
    ylabel=r'$\mathrm{Var}_{q_\phi}/\mathrm{Var}_{\rm NIS}$',
    title='Approximate event-saving factor',
)
fig.tight_layout()
export_nis_figure(fig, 'nis_asimov_convergence')
plt.show()

fig, axis = plt.subplots(figsize=(7.0, 5.0))
for method in ('Direct reference', 'Neural importance'):
    for index, curve in enumerate(showcase_scans[method]):
        axis.plot(
            MU_SCAN,
            curve,
            color=colors[method],
            alpha=0.22,
            lw=1.2,
            label=method if index == 0 else None,
        )
axis.plot(
    MU_SCAN,
    benchmark_scan,
    color='black',
    lw=2.6,
    label='Benchmark',
)
axis.axvline(1.0, color='0.5', ls='--', lw=1.0)
axis.set(
    xlabel=r'$\mu$',
    ylabel=r'$t_A(\mu)$',
    title=f'Repeated {SHOWCASE_SIZE:,}-point Asimov scans',
)
axis.legend()
fig.tight_layout()
export_nis_figure(fig, 'nis_repeated_small_asimov_scans')
plt.show()

selected = study_results[
    study_results['sample_size'] == SHOWCASE_SIZE
]
data = [
    selected.loc[selected['method'] == method, 'q_zero'].to_numpy()
    for method in ('Direct reference', 'Neural importance')
]
fig, axis = plt.subplots(figsize=(6.5, 4.8))
box = axis.boxplot(
    data,
    patch_artist=True,
    showmeans=True,
)
axis.set_xticks([1, 2], ['Direct reference', 'Neural importance'])
for patch, color in zip(
    box['boxes'],
    (DIRECT_COLOR, NIS_COLOR),
    strict=True,
):
    patch.set_facecolor(color)
    patch.set_alpha(0.35)
axis.axhline(
    benchmark_q0,
    color='black',
    lw=2,
    label='Benchmark',
)
axis.set(
    ylabel=r'$q_{0,A}$',
    title=f'Equal-cost comparison with {SHOWCASE_SIZE:,} points',
)
axis.legend()
fig.tight_layout()
export_nis_figure(fig, 'nis_q0_equal_cost')
plt.show()